# Stage 3: multi-model scientific hypothesis development

This notebook sends the same frozen aggregate Stage 2 evidence to every configured model. Each model produces standard, concise scientific hypotheses and a separate list of validity threats. Every scientific hypothesis is retained in a common ledger; exact duplicates are marked rather than removed.

The hypotheses are retrospective, exploratory, and untested. Policy-tree variables motivate subgroup hypotheses but do not establish moderation. Model priority is recorded for traceability; Stage 4 performs independent literature-informed evaluation and prepares expert review.


In [1]:
from pathlib import Path
from datetime import datetime
import csv, hashlib, json, os, re, shutil, subprocess, time, urllib.request

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/stage1_specifications.json').is_file())
OLLAMA=os.environ.get('OLLAMA_HOST','http://127.0.0.1:11434')
PREFERRED_LOCAL=['gpt-oss:20b','qwen3:30b','qwen3:14b','deepseek-r1:14b']
SEED=int(os.environ.get('REPLICATION_S3_SEED','20260923'))

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def api(path,payload=None,timeout=2400,retries=3):
    data=None if payload is None else json.dumps(payload).encode()
    request=urllib.request.Request(OLLAMA+path,data=data,headers={'Content-Type':'application/json'})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(request,timeout=timeout) as response:return json.load(response)
        except (OSError,json.JSONDecodeError):
            if attempt+1==retries:raise
            time.sleep(5*(attempt+1))

def latest_stage2():
    for p in sorted((ROOT/'S2/outputs').glob('S2_01_evidence_review_*/manifest.json'),reverse=True):
        manifest=json.loads(p.read_text()); bundle=p.parent/'evidence_bundle.json'
        if manifest.get('status')=='complete_retrospective_exploratory' and bundle.is_file() and sha256(bundle)==manifest['files']['evidence_bundle.json']:
            return p,bundle,json.loads(bundle.read_text())
    raise FileNotFoundError('No verified Stage 2 evidence bundle')

s2_manifest_path,bundle_path,bundle=latest_stage2()
try:
    installed={x.get('name') or x.get('model'):x for x in api('/api/tags')['models']}
except Exception as exc:
    raise ConnectionError(f'Cannot reach Ollama at {OLLAMA}. Start the Ollama server that points to the intended model store before running S3.') from exc
requested=[x.strip() for x in os.environ.get('REPLICATION_S3_LOCAL_MODELS',','.join(PREFERRED_LOCAL)).split(',') if x.strip()]
models=[x for x in requested if x in installed]
if os.environ.get('REPLICATION_S3_INCLUDE_CLAUDE','1')=='1' and shutil.which('claude'): models.append('claude-sonnet-4-6')
assert models,'No configured local model is available and Claude CLI was not enabled'
print('Evidence:',bundle_path.relative_to(ROOT)); print('Models:',models); print('Unavailable local models:',[x for x in requested if x not in installed])

Evidence: notebooks/code_0927/S2/outputs/S2_01_evidence_review_20260927_142855_247162/evidence_bundle.json
Models: ['gpt-oss:20b', 'qwen3:30b', 'qwen3:14b', 'deepseek-r1:14b', 'claude-sonnet-4-6']
Unavailable local models: []


In [2]:
HYPOTHESIS_TYPES=['overall_service_setting','tree_defined_heterogeneity']
TREE_SCOPES=['overall','global','local','emergent']
VALIDITY_THREAT_TYPES=['treatment_selection_or_residual_confounding','endpoint_measurement_or_state_reporting']
MECHANISM_DOMAINS=['service_organization_and_continuity','race_and_structural_access','referral_pathway','employment_and_schedule_fit','primary_substance_and_route','polysubstance_complexity']
EVIDENCE_IDS=[x['evidence_id'] for x in bundle['evidence_objects']]
compact=[{k:x[k] for k in ['evidence_id','analysis','treatment','outcome','target','heldout_n','heldout_states','finding','tree_pattern','quantities','limits']} for x in bundle['evidence_objects']]
hypothesis_properties={
 'hypothesis_id':{'type':'string'},'development_priority':{'type':'integer','minimum':1,'maximum':12},
 'analysis':{'type':'string','enum':['residential','outpatient']},'evidence_id':{'type':'string','enum':EVIDENCE_IDS},
 'hypothesis_type':{'type':'string','enum':HYPOTHESIS_TYPES},'tree_scope':{'type':'string','enum':TREE_SCOPES},'statement_id':{'type':'string'},'node_sequence':{'type':'string'},'mechanism_domain':{'type':'string','enum':MECHANISM_DOMAINS},
 'direction':{'type':'string','enum':['higher','lower']},'hypothesis':{'type':'string'},'mechanism':{'type':'string'},
 'tree_link':{'type':'string'},'observable_implication':{'type':'string'},'distinguishing_evidence':{'type':'string'},
 'failure_condition':{'type':'string'},'confidence':{'type':'string','enum':['low','moderate']}
}
threat_properties={
 'threat_id':{'type':'string'},'priority':{'type':'integer','minimum':1,'maximum':6},
 'analysis':{'type':'string','enum':['residential','outpatient','both']},'evidence_id':{'type':'string','enum':EVIDENCE_IDS},
 'threat_type':{'type':'string','enum':VALIDITY_THREAT_TYPES},'statement':{'type':'string'},
 'why_it_matters':{'type':'string'},'distinguishing_evidence':{'type':'string'}
}
schema={'type':'object','additionalProperties':False,'required':['study_status','model','hypotheses','validity_threats','cross_pattern_synthesis','citation_status'],'properties':{
 'study_status':{'type':'string','enum':['retrospective_exploratory']},'model':{'type':'string'},
 'hypotheses':{'type':'array','minItems':6,'maxItems':12,'items':{'type':'object','additionalProperties':False,'required':list(hypothesis_properties),'properties':hypothesis_properties}},
 'validity_threats':{'type':'array','minItems':2,'maxItems':6,'items':{'type':'object','additionalProperties':False,'required':list(threat_properties),'properties':threat_properties}},
 'cross_pattern_synthesis':{'type':'string'},'citation_status':{'type':'string','enum':['no_external_sources_supplied_or_verified']}}}
prompt=('Return one JSON object and no prose outside it. Develop 6 to 12 distinct scientific hypotheses from the supplied policy-tree evidence, with at least three for each analysis. Rank them by development_priority from 1 to N. Separately list 2 to 6 validity threats; confounding, treatment selection, and endpoint recording are rival validity explanations, not final scientific hypotheses. '
 'Write each hypothesis as one clear, succinct, paper-ready sentence beginning with H and its number, followed by a colon. For overall_service_setting use: H#: [treatment setting] is associated with a higher or lower probability of recorded treatment completion than [comparison setting]. For tree_defined_heterogeneity use: H#: Among [plain-language tree-defined subgroup], [treatment setting] is associated with a higher or lower probability of recorded treatment completion than [comparison setting]. Residential hypotheses must name long-term and short-term residential care. Outpatient hypotheses must name intensive and non-intensive outpatient care. Set direction to the same word used in the sentence. Put reasoning only in mechanism. '
 'Use the supplied tree scopes explicitly. overall uses the root node A and statement_id overall. global uses one G# full root-to-leaf path. local uses one L# split and its two branches, including hypotheses motivated by higher nodes. emergent uses one E# multi-node subtree pattern. Across the full response include every tree_scope, and for each analysis include at least one global hypothesis plus at least one local or emergent hypothesis. Copy statement_id and node_sequence exactly from tree_pattern; use tree_link to explain the plain-language link without repeating those structured fields. '
 'Do not put overlap population, overlap weighting, sample restrictions, variable codes, node letters, statement IDs, or REASON=1 in the hypothesis sentence; those technical definitions belong in the evidence-link fields. A tree_defined_heterogeneity hypothesis must name a subgroup using only plain-language codebook labels and split variables supplied in tree_pattern. The tree motivates the subgroup; it does not confirm moderation. Never call a personalized policy a treatment or an observed patient group. '
 'Use only supplied evidence. Recorded completion is REASON=1, not clinical recovery. These are retrospective observational estimates, not established causal effects. Treat every policy-tree result as an empirical pattern requiring explanation, not as a Kuhnian anomaly. A literature gap is not an anomaly. Do not claim contradiction with prevailing theory because Stage 3 receives no verified literature establishing such a conflict. Preserve the residential direction that long-term has lower estimated completion in every held-out leaf. Preserve the outpatient finding that the estimated policy value exceeds both constant policies while always-intensive is below always-non-intensive; this may reflect heterogeneity or estimation bias and is not an average intensive-care benefit. '
 'Do not invent subgroups, variables, observed policy assignments, effects, literature, citations, or statistical significance. A scientific mechanism must describe a substantive process and must not merely restate confounding or measurement. Distinguishing evidence must add information beyond the supplied finding. A failure condition must genuinely contradict the hypothesis. Do not recommend treatment. Follow this schema exactly: '+json.dumps(schema,separators=(',',':'))+' EVIDENCE: '+json.dumps(compact,separators=(',',':')))


In [3]:
def slug(value): return re.sub(r'[^a-z0-9]+','_',value.lower()).strip('_')

def parse_json_text(text):
    return json.loads(re.sub(r'^```(?:json)?\s*|\s*```$','',text.strip()))

def model_prompt(model):
    effective=prompt.replace('"model":{"type":"string"}',f'"model":{{"type":"string","const":"{model}"}}')
    return '/no_think\n'+effective if model.startswith('qwen3:') else effective

def run_model(model,effective):
    model_schema=json.loads(json.dumps(schema));model_schema['properties']['model']={'type':'string','const':model}
    started=time.perf_counter()
    if model.startswith('claude-'):
        command=['claude','-p','--model',model,'--effort','medium','--tools','','--no-session-persistence','--output-format','json','--json-schema',json.dumps(model_schema,separators=(',',':'))]
        process=subprocess.run(command,input=effective,text=True,capture_output=True,check=False)
        if process.returncode: raise RuntimeError(process.stderr[-2000:])
        raw=json.loads(process.stdout); response=raw.get('structured_output') or parse_json_text(raw.get('result',''))
        usage=raw.get('usage',{}); runtime={'seconds':time.perf_counter()-started,'temperature':'not_exposed_by_claude_cli','seed':'not_exposed_by_claude_cli','cost_usd':raw.get('total_cost_usd'),'input_tokens':usage.get('input_tokens',0),'cache_creation_input_tokens':usage.get('cache_creation_input_tokens',0),'cache_read_input_tokens':usage.get('cache_read_input_tokens',0),'output_tokens':usage.get('output_tokens',0),'resolved_models':list(raw.get('modelUsage',{}))}
        return response,runtime,{'stdout':raw,'stderr':process.stderr}
    payload={'model':model,'messages':[{'role':'user','content':effective}], 'stream':False,'format':model_schema,'options':{'temperature':0,'seed':SEED,'top_p':1,'num_ctx':32768,'num_predict':7000},'keep_alive':'0'}
    if model.startswith('qwen3:'): payload['think']=False
    raw=api('/api/chat',payload);content=raw.get('message',{}).get('content','')
    if not content.strip():
        thinking=raw.get('message',{}).get('thinking','')
        raise ValueError(f'Model returned empty final content; thinking_characters={len(thinking)}, done_reason={raw.get("done_reason")}.')
    response=parse_json_text(content)
    runtime={'seconds':time.perf_counter()-started,'temperature':0,'seed':SEED,'prompt_tokens':raw.get('prompt_eval_count'),'output_tokens':raw.get('eval_count'),'done_reason':raw.get('done_reason')}
    return response,runtime,raw

def normalized_text(value):
    return str(value).lower().translate(str.maketrans({'‐':'-','‑':'-','‒':'-','–':'-','—':'-','−':'-'}))

statement_index={}
for evidence in bundle['evidence_objects']:
    analysis=evidence['analysis'];statement_index[(analysis,'overall','overall')]='A'
    for scope,records_ in evidence['tree_pattern']['statements'].items():
        for record in records_:statement_index[(analysis,scope,record['statement_id'])]=record['node_sequence']

def validate(response,model):
    hs=response.get('hypotheses',[]); threats=response.get('validity_threats',[]); text=normalized_text(json.dumps(response,ensure_ascii=False)); analyses=[h.get('analysis') for h in hs]
    item_records=[]
    for h in hs:
        claim=normalized_text(h.get('hypothesis',''));analysis=h.get('analysis');direction=h.get('direction');hypothesis_type=h.get('hypothesis_type');tree_scope=h.get('tree_scope');statement_id=h.get('statement_id');node_sequence=h.get('node_sequence')
        item_checks={
          'known_analysis_and_evidence':(analysis,h.get('evidence_id')) in {('residential','residential_global_01'),('outpatient','outpatient_policy_01')},
          'standard_hypothesis_prefix':bool(re.match(r'^h\d+\s*:',claim)),
          'direction_field_valid':direction in {'higher','lower'},
          'direction_stated_in_hypothesis':direction in claim if direction else False,
          'recorded_completion_named':any(x in claim for x in ['recorded treatment completion','recorded completion']),
          'paper_ready_wording':not any(x in claim for x in ['overlap population','overlap-weighted','reason=1','reason = 1']),
          'treatment_contrast_named':(('long-term' in claim and 'short-term' in claim and 'residential' in claim) if analysis=='residential' else ('intensive' in claim and 'non-intensive' in claim and 'outpatient' in claim) if analysis=='outpatient' else False),
          'association_wording':'associated with' in claim,
          'no_personalized_policy_as_treatment':'personalized policy' not in claim and 'personalized outpatient treatment' not in claim,
          'scientific_type_only':hypothesis_type in HYPOTHESIS_TYPES,
          'tree_scope_matches_type':(tree_scope=='overall')==(hypothesis_type=='overall_service_setting'),
          'statement_reference_exact':statement_index.get((analysis,tree_scope,statement_id))==node_sequence,
          'empirical_direction_preserved':direction=='lower' if analysis=='residential' or (analysis=='outpatient' and hypothesis_type=='overall_service_setting') else True,
          'no_double_negation':'not not' not in claim,
          'confidence_valid':h.get('confidence') in {'low','moderate'},
        }
        item_checks['passed']=all(item_checks.values()); item_records.append({'hypothesis_id':h.get('hypothesis_id'),'checks':item_checks})
    threat_types={x.get('threat_type') for x in threats}
    checks={
      'top_level_fields_exact':set(response)=={'study_status','model','hypotheses','validity_threats','cross_pattern_synthesis','citation_status'},
      'study_status_exact':response.get('study_status')=='retrospective_exploratory','model_exact':response.get('model')==model,
      'candidate_count_6_to_12':6<=len(hs)<=12,'unique_hypothesis_ids':len({h.get('hypothesis_id') for h in hs})==len(hs),
      'priority_sequence_exact':sorted(h.get('development_priority') for h in hs)==list(range(1,len(hs)+1)),
      'both_analyses_covered':analyses.count('residential')>=3 and analyses.count('outpatient')>=3,
      'both_hypothesis_types_represented':set(HYPOTHESIS_TYPES)<=set(h.get('hypothesis_type') for h in hs),
      'all_tree_scopes_represented':set(TREE_SCOPES)<=set(h.get('tree_scope') for h in hs),
      'each_analysis_has_global_and_higher_or_group_scope':all(any(h.get('analysis')==a and h.get('tree_scope')=='global' for h in hs) and any(h.get('analysis')==a and h.get('tree_scope') in {'local','emergent'} for h in hs) for a in ['residential','outpatient']),
      'validity_threat_count_2_to_6':2<=len(threats)<=6,
      'validity_threats_separate':set(VALIDITY_THREAT_TYPES)<=threat_types,
      'no_significance_language':'signific' not in text,'no_treatment_recommendation':not any(x in text for x in ['should receive','recommend treatment','should be assigned']),
      'citation_status_exact':response.get('citation_status')=='no_external_sources_supplied_or_verified'}
    checks['response_structure_passed']=all(checks.values())
    checks['hypothesis_validation']=item_records
    checks['all_hypotheses_passed']=bool(item_records) and all(x['checks']['passed'] for x in item_records)
    checks['structural_and_fidelity_gate_passed']=checks['response_structure_passed'] and checks['all_hypotheses_passed']
    return checks


In [4]:
stamp=datetime.now().astimezone().strftime('%Y%m%d_%H%M%S_%f')
out=ROOT/'S3/outputs'/f'S3_01_hypothesis_generation_{stamp}'; out.mkdir(parents=True)
records=[]
for model in models:
    model_dir=out/slug(model);model_dir.mkdir();effective=model_prompt(model);(model_dir/'prompt.txt').write_text(effective)
    raw={}
    try:
        response,runtime,raw=run_model(model,effective);checks=validate(response,model)
    except Exception as error:
        response={}; runtime={'error':repr(error)}; checks={'structural_and_fidelity_gate_passed':False,'execution_error':repr(error)}
    (model_dir/'raw_response.json').write_text(json.dumps(raw,indent=2));(model_dir/'model_response.json').write_text(json.dumps(response,indent=2))
    (model_dir/'runtime.json').write_text(json.dumps(runtime,indent=2)); (model_dir/'validation.json').write_text(json.dumps(checks,indent=2))
    records.append({'model':model,'model_dir':str(model_dir.relative_to(out)),'response':response,'runtime':runtime,'validation':checks})
    print(model,checks['structural_and_fidelity_gate_passed'],len(response.get('hypotheses',[])))

gpt-oss:20b False 8
qwen3:30b False 6
qwen3:14b False 6
deepseek-r1:14b False 12
claude-sonnet-4-6 True 10


In [5]:
def normalized_claim(value): return re.sub(r'[^a-z0-9]+',' ',value.lower()).strip()
all_candidates=[]; all_threats=[]
for record in records:
    item_validation={x['hypothesis_id']:x['checks'] for x in record['validation'].get('hypothesis_validation',[])}
    response_ok=record['validation'].get('response_structure_passed',False)
    for h in record['response'].get('hypotheses',[]):
        h_checks=item_validation.get(h['hypothesis_id'],{'passed':False})
        all_candidates.append({'candidate_id':slug(record['model'])+'::'+h['hypothesis_id'],'candidate_role':'scientific_hypothesis','model':record['model'],'model_validation_passed':record['validation']['structural_and_fidelity_gate_passed'],'hypothesis_validation_passed':h_checks.get('passed',False),'hypothesis_validation':h_checks,**h})
    for t in record['response'].get('validity_threats',[]):
        all_threats.append({'threat_id':slug(record['model'])+'::'+t['threat_id'],'candidate_role':'validity_threat','model':record['model'],**t})
groups={}
for h in all_candidates: groups.setdefault(normalized_claim(h['hypothesis']),[]).append(h['candidate_id'])
for h in all_candidates:
    members=groups[normalized_claim(h['hypothesis'])]; h['exact_duplicate_group']=members if len(members)>1 else []

hypothesis_fields=['candidate_id','hypothesis_id','candidate_role','model','model_validation_passed','hypothesis_validation_passed','development_priority','analysis','evidence_id','hypothesis_type','tree_scope','statement_id','node_sequence','mechanism_domain','hypothesis','mechanism','tree_link','observable_implication','distinguishing_evidence','failure_condition','confidence','exact_duplicate_group']
with (out/'generated_hypotheses.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=hypothesis_fields,extrasaction='ignore'); writer.writeheader()
    for h in all_candidates: writer.writerow({**h,'exact_duplicate_group':json.dumps(h['exact_duplicate_group'])})
threat_fields=['threat_id','candidate_role','model','priority','analysis','evidence_id','threat_type','statement','why_it_matters','distinguishing_evidence']
with (out/'validity_threats.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=threat_fields,extrasaction='ignore');writer.writeheader();writer.writerows(all_threats)

def md(value): return str(value).replace('|','\\|').replace('\n',' ')
md_lines=['# Stage 3 scientific hypotheses','','All hypotheses are retrospective, exploratory, and untested. Confounding and endpoint-recording explanations are reported separately as validity threats.','', '| Model | Priority | Analysis | Scope | Statement | Nodes | Hypothesis | Validation | Duplicate group |','|---|---:|---|---|---|---|---|---|---|']
for h in sorted(all_candidates,key=lambda x:(x['model'],x['development_priority'])):
    duplicate=', '.join(h['exact_duplicate_group']) if h['exact_duplicate_group'] else ''
    md_lines.append('| '+' | '.join(md(x) for x in [h['model'],h['development_priority'],h['analysis'],h['tree_scope'],h['statement_id'],h['node_sequence'],h['hypothesis'],'passed' if h['hypothesis_validation_passed'] else 'failed',duplicate])+' |')
(out/'generated_hypotheses.md').write_text('\n'.join(md_lines)+'\n')
threat_lines=['# Stage 3 validity threats','','These are rival explanations and measurement concerns, not final scientific hypotheses.','', '| Model | Priority | Analysis | Type | Statement |','|---|---:|---|---|---|']
for t in sorted(all_threats,key=lambda x:(x['model'],x['priority'])): threat_lines.append('| '+' | '.join(md(x) for x in [t['model'],t['priority'],t['analysis'],t['threat_type'],t['statement']])+' |')
(out/'validity_threats.md').write_text('\n'.join(threat_lines)+'\n')
ledger={'artifact_status':'stage3_candidate_ledger_ready_for_stage4','created_at':datetime.now().astimezone().isoformat(),'source_stage2':str(bundle_path.relative_to(ROOT)),'source_stage2_sha256':sha256(bundle_path),'retention_rule':'all generated scientific hypotheses retained; exact duplicates marked; validity threats stored separately','models':[{'model':r['model'],'model_dir':r['model_dir'],'validation':r['validation'],'runtime':r['runtime']} for r in records],'candidates':all_candidates,'validity_threats':all_threats,'candidate_count':len(all_candidates),'validity_threat_count':len(all_threats),'stage4_eligible_count':sum(h['hypothesis_validation_passed'] for h in all_candidates)}
(out/'candidate_ledger.json').write_text(json.dumps(ledger,indent=2))
files={str(p.relative_to(out)):sha256(p) for p in out.rglob('*') if p.is_file() and p.name!='manifest.json'}
manifest={'status':'complete_candidate_ledger_researcher_and_stage4_review_required','created_at':ledger['created_at'],'source_stage2':ledger['source_stage2'],'source_stage2_sha256':ledger['source_stage2_sha256'],'models_requested':models,'model_runs':len(records),'candidate_count':ledger['candidate_count'],'validity_threat_count':ledger['validity_threat_count'],'stage4_eligible_count':ledger['stage4_eligible_count'],'local_options':{'temperature':0,'seed':SEED,'top_p':1},'hosted_options':{'effort':'medium','temperature':'not_exposed','seed':'not_exposed'},'files':files,'limits':['Every hypothesis is untested.','Tree splits motivate subgroup hypotheses but do not establish moderation.','Structural validation does not establish scientific quality.','Validity threats are not final hypotheses.','Researcher and domain-expert review are required.']}
(out/'manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps({k:manifest[k] for k in ['status','candidate_count','validity_threat_count','stage4_eligible_count']},indent=2)); print('Output:',out.relative_to(ROOT))


{
  "status": "complete_candidate_ledger_researcher_and_stage4_review_required",
  "candidate_count": 42,
  "validity_threat_count": 17,
  "stage4_eligible_count": 33
}
Output: notebooks/code_0927/S3/outputs/S3_01_hypothesis_generation_20260927_143310_956207


In [6]:
# Export the Stage 3 hypotheses and separate validity-threat ledger to Word.
from html import escape as html_escape

assert shutil.which('pandoc'),'pandoc is required for the Word export'
def html_cell(value,header=False):
    tag='th' if header else 'td'
    return f'<{tag}>{html_escape(str(value))}</{tag}>'
def html_table(headers,rows):
    head='<tr>'+''.join(html_cell(x,True) for x in headers)+'</tr>'
    body=''.join('<tr>'+''.join(html_cell(x) for x in row)+'</tr>' for row in rows)
    return '<table>'+head+body+'</table>'

hypothesis_rows=[]; detail_sections=[]
for h in sorted(all_candidates,key=lambda x:(x['model'],x['development_priority'])):
    failed=', '.join(k for k,v in h.get('hypothesis_validation',{}).items() if v is False and k!='passed') or 'None'
    eligibility='eligible for S4 ranking' if h['hypothesis_validation_passed'] else 'flagged for revision'
    hypothesis_rows.append([h['candidate_id'],h['model'],h['development_priority'],h['analysis'],h['tree_scope'],h['statement_id'],h['node_sequence'],h['hypothesis'],eligibility,failed])
    detail_sections.append('<h3>'+html_escape(h['candidate_id'])+'</h3>'+html_table(['Field','Content'],[
        ['Hypothesis',h['hypothesis']],['Scope',h['tree_scope']],['Statement ID',h['statement_id']],['Node sequence',h['node_sequence']],['Mechanism',h['mechanism']],['Tree link',h['tree_link']],
        ['Observable implication',h['observable_implication']],['Distinguishing evidence',h['distinguishing_evidence']],
        ['Failure condition',h['failure_condition']],['S3 status',eligibility],['Failed checks',failed]
    ]))
threat_rows=[[t['threat_id'],t['model'],t['priority'],t['analysis'],t['threat_type'],t['statement'],t['why_it_matters'],t['distinguishing_evidence']] for t in sorted(all_threats,key=lambda x:(x['model'],x['priority']))]
word_html=f'''<html><head><meta charset="utf-8"><style>body{{font-family:Arial;font-size:10pt}} table{{border-collapse:collapse;width:100%;margin-bottom:12px}} th,td{{border:1px solid #888;padding:4px;vertical-align:top}} th{{background:#d9eaf7}} h1,h2,h3{{page-break-after:avoid}}</style></head><body><h1>Stage 3 Generated Hypotheses</h1><p>All hypotheses are retrospective, exploratory, and untested. S3 eligibility reflects structural and evidence-fidelity checks, not scientific support or truth.</p><h2>Hypotheses</h2>{html_table(['Candidate','Model','Priority','Analysis','Scope','Statement','Nodes','Hypothesis','S3 status','Failed checks'],hypothesis_rows)}<h2>Hypothesis details</h2>{''.join(detail_sections)}<h2>Validity threats</h2><p>These are rival explanations and measurement concerns, not final scientific hypotheses.</p>{html_table(['Threat','Model','Priority','Analysis','Type','Statement','Why it matters','Distinguishing evidence'],threat_rows)}</body></html>'''
word_path=out/'generated_hypotheses.docx'
temp_html=out/'.generated_hypotheses_tmp.html'
temp_html.write_text(word_html)
process=subprocess.run(['pandoc',str(temp_html),'-o',str(word_path)],text=True,capture_output=True,check=False)
temp_html.unlink(missing_ok=True)
if process.returncode: raise RuntimeError(process.stderr)
index_lines=['# Coauthor review index','','Exact model prompts and complete responses are retained so prompt wording and results can be reviewed together.','']
for record in records:
    folder=record['model_dir'];index_lines.extend([f"## {record['model']}",'',f'- Exact submitted prompt: `{folder}/prompt.txt`',f'- Structured response: `{folder}/model_response.json`',f'- Raw response: `{folder}/raw_response.json`',f'- Validation: `{folder}/validation.json`',f'- Runtime and model settings: `{folder}/runtime.json`',''])
index_lines.extend(['## Combined results','', '- `generated_hypotheses.docx`', '- `generated_hypotheses.csv`', '- `generated_hypotheses.md`', '- `validity_threats.md`', '- `candidate_ledger.json`',''])
index_path=out/'COAUTHOR_REVIEW_INDEX.md';index_path.write_text('\n'.join(index_lines))
manifest=json.loads((out/'manifest.json').read_text());manifest['files']['generated_hypotheses.docx']=sha256(word_path);manifest['files']['COAUTHOR_REVIEW_INDEX.md']=sha256(index_path);(out/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Word report:',word_path.relative_to(ROOT),'Hypotheses:',len(all_candidates),'Validity threats:',len(all_threats))


Word report: notebooks/code_0927/S3/outputs/S3_01_hypothesis_generation_20260927_143310_956207/generated_hypotheses.docx Hypotheses: 42 Validity threats: 17


## Human review gate

Stage 4 evaluates every structurally valid scientific hypothesis automatically. Treatment selection, residual confounding, and endpoint recording remain in the separate validity-threat ledger and accompany expert review. Before coauthor use, a researcher must still check treatment direction, tree-path fidelity, codebook labels, redundancy, and causal language. Subsequent domain-expert review can assess clinical plausibility and theoretical value.
